In [1]:
import pandas as pd
from utils import bootstrap_filter, get_clust
from GenFocus import genfocus

c:\Users\dyuga\miniconda3\envs\llm\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load data from disc.

In [2]:
data = pd.read_csv('data/TCGA_BLCA.tsv', sep='\t', index_col=0, header=0)

Drop genes with constant expression.

In [3]:
data = data.loc[:, (data != data.iloc[0]).any()]

Drop genes with identical expression (the first one will be kept).

In [4]:
data = data.T.drop_duplicates().T

Drop genes with classification error less than $\frac{1}{e}$. If you have two datasets, clear both of them and use the intersection of genes.

In [5]:
clear_data = bootstrap_filter(data).copy()

The main hyperparameter to tune is the **minimum cluster size**.
The method returns two arrays with cluster labels for each gene. The first one contains clusters from the HDBSCAN. The second contains same clusters but eroded by the core decomposition. They will be much denser and smaller. The label -1 is assigned to noize genes outside any cluster.

In [6]:
clusters, core_clusters = get_clust(clear_data, min_clust_size=100, n_components=6)

c:\Users\dyuga\miniconda3\envs\llm\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\dyuga\miniconda3\envs\llm\lib\site-packages\umap\umap_.py:1865: UserWarning: using precomputed metric; inverse_transform will be unavailable
  warn("using precomputed metric; inverse_transform will be unavailable")
c:\Users\dyuga\miniconda3\envs\llm\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
c:\Users\dyuga\miniconda3\envs\llm\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
c:\Users\dyuga\miniconda3\envs\llm\lib\site-packages\sklearn\utils\deprecation.py:151: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be remove

In [7]:
count_modules = {}
for i in clusters:
    count_modules[i] = count_modules.get(i, 0) + 1
count_modules

{np.int64(-1): 10646,
 np.int64(8): 138,
 np.int64(0): 107,
 np.int64(24): 1901,
 np.int64(6): 215,
 np.int64(2): 274,
 np.int64(17): 332,
 np.int64(16): 119,
 np.int64(21): 463,
 np.int64(15): 685,
 np.int64(10): 141,
 np.int64(3): 196,
 np.int64(19): 235,
 np.int64(13): 247,
 np.int64(9): 166,
 np.int64(22): 102,
 np.int64(18): 124,
 np.int64(7): 365,
 np.int64(11): 123,
 np.int64(1): 129,
 np.int64(4): 135,
 np.int64(12): 220,
 np.int64(5): 149,
 np.int64(14): 168,
 np.int64(23): 129,
 np.int64(20): 102}

In [8]:
signatures = {}

with open("data/bostongene_signatures.txt") as f:
    for line in f.readlines():
        signatures[line.split("\t")[0]+'_bostongen'] = line.strip().split("\t")[2:]

with open("data/xCell_signatures.csv") as f:
    for line in f.readlines()[1:]:
        signatures[line.split(",")[0]] = list(filter(None, line.strip().split(",")[2:]))

In [9]:
angio = signatures['Angiogenesis_bostongen']
cd8 = ['CD8A', 'CXCL10', 'CXCL9', 'GZMA', 'GZMB', 'IFNG', 'PRF1', 'TBX21', 'FOXP3', 'CTLA4', 'GZMK', 'CD4', 'PTPRC', 'CD83']
caf_genes = ['ACTA2', 'FAP', 'POSTN', 'ASPN', 'MFAP5', 'PDPN', 'ITGA11', 'PDGFRA', 'PDGFRB', 'S100A4', 'VIM', 'COL11A1', 'TGFB1']

modules = pd.DataFrame({'module':clusters}, index=clear_data.columns)
for i in caf_genes:
    print(i, modules.loc[i, 'module'])

ACTA2 19
FAP 18
POSTN 18
ASPN 18
MFAP5 3
PDPN -1
ITGA11 -1
PDGFRA 8
PDGFRB 18
S100A4 -1
VIM 9
COL11A1 18
TGFB1 -1


In [10]:
import gseapy as gp

modules = pd.DataFrame({'module':clusters}, index=clear_data.columns)
unique_modules = sorted(modules['module'].unique())

for i in unique_modules:
    gene_list = list(modules[modules.module==i].index)
    if not gene_list:  
        continue
    enr = gp.enrich(gene_list=gene_list, 
                    gene_sets=signatures,
                    background=list(modules.index),
                    outdir=None,
                    verbose=False)
    enr_results = enr.results[enr.results['Adjusted P-value'] < 0.05].sort_values(by='Adjusted P-value')[['Term','Overlap','P-value', 'Adjusted P-value', 'Odds Ratio']]
    if len(enr_results)==0:
        continue
    print(f'Module {i} annotation, module size: {len(gene_list)}\n')
    print(enr_results)

Module -1 annotation, module size: 10646

                              Term Overlap   P-value  Adjusted P-value  \
159     Endothelial cells_ENCODE_1   45/50  0.000003          0.000911   
160     Endothelial cells_ENCODE_2   38/41  0.000004          0.000911   
484  mv Endothelial cells_FANTOM_3  85/107  0.000021          0.003607   
182      Epithelial cells_FANTOM_2   38/45  0.000452          0.045791   
394              Osteoblast_HPCA_3   46/56  0.000417          0.045791   

     Odds Ratio  
159    5.431526  
160    7.219447  
484    2.498253  
182    3.367140  
394    2.905800  
Module 0 annotation, module size: 107

                    Term Overlap   P-value  Adjusted P-value  Odds Ratio
17    Hepatocytes_HPCA_1   6/142  0.000229          0.003665    8.148487
19    Hepatocytes_HPCA_3   6/131  0.000148          0.003665    8.868310
34  Melanocytes_FANTOM_3   7/197  0.000200          0.003665    6.782818
32  Melanocytes_ENCODE_3    2/10  0.001594          0.019130   48.777530
M

In [11]:
import gseapy as gp

Normalizing a cluster using genfocus. In these example the 0th cluster.

In [12]:
modules = pd.DataFrame({'module':clusters}, index=clear_data.columns)
module = clear_data[modules[modules.loc[:, 'module']==0].index]
ings, eigen_focus_module = genfocus(module, focus_gene='eigengene', method='spearman', corr_thr=0.8, CVR_thr=0.5)

INGS selected by spearman correlation: 0


ZeroDivisionError: division by zero

Discovering fine-grained modules.

In [ ]:
subclusters, core_subclusters = get_clust(eigen_focus_module[~eigen_focus_module.isna().any(axis=1)].drop(ings, axis=1), min_clust_size=10, n_components=10)

Practical example on TCGA BLCA Dataset

In [ ]:
blca_tpm = pd.read_table('TCGA_BLCA_TPM.tsv', index_col=0, header=0)

In [ ]:
def immfocus(df, focus_gene='PTPRC', method='spearman', core=None, tpm=True, corr_thr=0.9, CVR_thr=0.6, power=1):
    
    if not tpm:
        ##создание таблицы TPM
        df['Total']=df.sum(axis = 1, skipna = True)
        df_tpm = ((df.loc[:, df.columns != 'Total'].astype(float).div(df.Total.astype(float), axis=0))*(10**6)).fillna(0)
    else:
        df_tpm = df.copy()
    df_ings = df_tpm.copy()
    
    if core is not None:
        df_ings = df_ings.loc[:, core]
    
    #спирман
    if focus_gene=='eigengene':
        
        mm = MinMaxScaler()
        rs = RobustScaler()
        trans_df = pd.DataFrame(data=rs.fit_transform(X=df_ings), index=df_ings.index, columns=df_ings.columns)
        trans_df = pd.DataFrame(data=mm.fit_transform(X=trans_df), index=trans_df.index, columns=trans_df.columns)

        pca = PCA(n_components=1)
        eigengene = pca.fit_transform(trans_df)
        eigengene = pd.Series(eigengene.T[0], index=df_ings.index)
        cor = df_ings.corrwith(eigengene, axis=0, method=method)
    else:
        cor = df_ings.corrwith(df_ings[focus_gene], axis=0, method=method)
        
    df_ings.loc['correlation'] = cor

    ings = df_ings.loc[:, df_ings.loc['correlation', :]>corr_thr].columns
    print(f'INGS selected by spearman correlation: {len(ings)}')
    for j in [i for i in ings]:
        print(j, end='\t')
    if focus_gene=='eigengene':
        if len(ings) == 1:
            print(f'No genes are correlate with {focus_gene} on correlation level {corr_thr}')
            return None
    else:
        if len(ings) == 0:
            print(f'No genes are correlate with {focus_gene} on correlation level {corr_thr}')
            return None

    #раасчет Fings для каждого образца
    fings = df_ings.apply(lambda x: sum(x[ings])/len(ings), axis=1)

    #нормализация
    df_ings_norm = df_tpm.copy()
    df_ings_norm.loc[:, exclusion(df_ings_norm.columns, ings)] = df_ings_norm.loc[:, exclusion(df_ings_norm.columns, ings)].div(fings, axis=0)
    for j in ings:
        fing = df_ings.apply((lambda x: sum(x[exclusion(ings, j)])/(len(ings)-1)), axis=1)
        df_ings_norm.loc[:, j] = df_ings_norm.loc[:, j].div(fing, axis=0)

    #вычисление CV
    for i in ings:
        if (df_ings.loc[df_tpm.index.to_list(), i].mean()==0) or (df_ings_norm.loc[df_tpm.index.to_list(), i].mean()==0):
            continue
        CV = df_ings.loc[df_tpm.index.to_list(), i].std()/df_ings.loc[df_tpm.index.to_list(), i].mean()
        df_ings.loc['CV', i] = CV

        CV = df_ings_norm.loc[df_tpm.index.to_list(), i].std()/df_ings_norm.loc[df_tpm.index.to_list(), i].mean()
        df_ings_norm.loc['CV', i] = CV

        df_ings_norm.loc['CVR', i] = df_ings_norm.loc['CV', i]/df_ings.loc['CV', i]

    #определение окончательной выборки генов INGS
    '''if core is not None:
        CVR_good = df_ings_norm.loc[:, df_ings_norm.loc['CVR', core]<CVR_thr].columns
    else:'''
    CVR_good = df_ings_norm.loc[:, df_ings_norm.loc['CVR', :]<CVR_thr].columns
    if len(CVR_good) == 0:
        return None
    ings = intersection(ings, CVR_good)
    print(f'\nINGS selected by CVR clipping: {len(ings)}')
    for j in [i for i in ings]:
        print(j, end='\t')

    #финальная нормализация
    fings = df_ings.apply(lambda x: sum(x[ings])/len(ings), axis=1)
    
    df_tpm.loc[:, exclusion(df_tpm.columns, ings)] = df_tpm.loc[:, exclusion(df_tpm.columns, ings)].div((fings**power), axis=0)
    for j in ings:
        fing = df_tpm.apply((lambda x: sum(x[exclusion(ings,j)])/(len(ings)-1)), axis=1)
        df_tpm.loc[:, j] = df_tpm.loc[:, j].div((fing**power), axis=0)
    
    return fings, ings, df_tpm